# F1 MultiCircuit Lap Time Model — Google Colab Runner

**Antes de começar:**
1. Vá em **Runtime → Change runtime type** e selecione **T4 GPU** (necessário para o LSTM).
2. Certifique-se de que o repositório está salvo no seu Google Drive em:
   `MyDrive/TCC/F1-MultiCircuit-LapTimeModel`
   (ou ajuste `REPO_PATH` na célula de configuração abaixo).
3. Execute as células em ordem.

## 1. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configurar path do repositório

In [ ]:
import os
import sys

# Ajuste este path para onde o repositório está no seu Drive
REPO_PATH = "/content/drive/MyDrive/TCC/F1-MultiCircuit-LapTimeModel"

assert os.path.isdir(REPO_PATH), f"Repositório não encontrado em: {REPO_PATH}"
os.chdir(REPO_PATH)
sys.path.insert(0, os.path.join(REPO_PATH, "Scripts", "Source"))
print(f"Diretório de trabalho: {os.getcwd()}")

## 3. Instalar dependências

> **Nota:** O TensorFlow já está disponível no Colab. As demais dependências serão instaladas via `requirements.txt`.

In [ ]:
!pip install -q -r Utils/requirements.txt
print("Dependências instaladas.")

## 4. Verificar GPU disponível

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU disponível: {gpus[0].name}")
else:
    print("AVISO: Nenhuma GPU detectada. O LSTM vai rodar em CPU (mais lento).")
    print("Para usar GPU: Runtime → Change runtime type → T4 GPU")

## 5. Rodar experimentos

Selecione o circuito e o(s) modelo(s) a rodar.

| Opção `MODELS` | Descrição |
|---|---|
| `lstm` | LSTM expanding-window |
| `xgb_ew` | XGBoost expanding-window |
| `lr_ew` | Linear Regression expanding-window |
| `xgb` | XGBoost sliding-window |
| `lr` | Linear Regression sliding-window |

Para rodar múltiplos modelos: `MODELS = "lr_ew xgb_ew lstm"`

> **Dica:** Após a primeira execução do LSTM, defina `use_saved_lstm_params: true` no YAML do circuito para reutilizar os params do Optuna e economizar tempo.

In [ ]:
# @title Configuração do experimento { run: "auto" }
GP     = "bahrain"  # @param ["bahrain", "saudi", "usa", "italy", "hungary", "all"]
MODELS = "lstm"     # @param ["lstm", "lr_ew", "xgb_ew", "lr", "xgb", "lr_ew xgb_ew lstm", "lr xgb"]

In [ ]:
gp_flag = "" if GP == "all" else f"--gp {GP}"
cmd = f"python Scripts/Source/run_all_models.py {gp_flag} --models {MODELS}"
print(f"Executando: {cmd}\n")
!{cmd}

## 6. Inspecionar resultados

In [ ]:
import json
from pathlib import Path

safe_name = GP.lower().replace(" ", "_")

# Metadados do LSTM EW
lstm_meta_path = Path(f"Scripts/Results/lstm/ew/models/{safe_name}_lstm_model_ew_metadata.json")
if lstm_meta_path.exists():
    meta = json.loads(lstm_meta_path.read_text())
    print("=== LSTM EW — Métricas de validação ===")
    for k, v in meta.get("val_metrics", {}).items():
        print(f"  {k}: {v:.4f}")
    print(f"\nSequence length: {meta.get('sequence_length')}")
    print(f"Final epoch count: {meta.get('final_epoch_count')}")
    if meta.get("optuna_summary"):
        print(f"Optuna best RMSE: {meta['optuna_summary'].get('best_value', 'N/A'):.4f}")
        print(f"Optuna best params: {meta['optuna_summary'].get('best_params')}")
else:
    print(f"Metadata não encontrado em: {lstm_meta_path}")

# Trials CSV do Optuna
trials_path = Path(f"Scripts/Results/lstm/ew/params/{safe_name}_lstm_optuna_trials_ew.csv")
if trials_path.exists():
    import pandas as pd
    trials_df = pd.read_csv(trials_path)
    print(f"\n=== Optuna trials ({len(trials_df)} total) — top 5 por RMSE ===")
    print(trials_df.sort_values("rmse").head(5).to_string(index=False))

## 7. (Opcional) Visualizar experimentos no MLflow

> Execute esta célula para abrir o MLflow UI via túnel ngrok (requer conta gratuita em ngrok.com).

In [ ]:
# Instalar e iniciar ngrok (só necessário se quiser o MLflow UI)
!pip install -q pyngrok
from pyngrok import ngrok
import subprocess, time

MLFLOW_URI = "Scripts/Results/mlruns"
proc = subprocess.Popen(["mlflow", "ui", "--backend-store-uri", MLFLOW_URI, "--port", "5000"])
time.sleep(3)
tunnel = ngrok.connect(5000)
print(f"MLflow UI disponível em: {tunnel.public_url}")